# Support Vector Machine

Finds the **maximum margin hyperplane**, where "margin" refers to the distance between the hyperplane and the closest data points (called **support vectors**).

<br>

<p align="center">
<img src="visualizations/SVM.png" width="600">
</p>

## Mathematical Foundation

### Linear SVM (Linearly Separable Data)

Given labeled data:
- Features: $\mathbf{x}_i \in \mathbb{R}^n$  
- Labels: $y_i \in \{-1, +1\}$

SVM tries to find a hyperplane defined by:
$$
\mathbf{w}^\top \mathbf{x} + b = 0
$$

That separates the two classes while maximizing the margin $\frac{2}{||\mathbf{w}||}$.

**Optimization Problem:**
$$
\min_{\mathbf{w}, b} \frac{1}{2} ||\mathbf{w}||^2
$$

Subject to:
$$
y_i(\mathbf{w}^\top \mathbf{x}_i + b) \geq 1
$$

The data points that lie on the margin boundary (where $y_i(\mathbf{w}^\top \mathbf{x}_i + b) = 1$) are called **support vectors**.

### Non-linearly Separable Data & Soft Margin

When data isn't perfectly separable, SVM uses a **soft margin** that allows some misclassifications:
$$
\min_{\mathbf{w}, b, \xi} \frac{1}{2} ||\mathbf{w}||^2 + C \sum_{i=1}^{n} \xi_i
$$

Subject to:
$$
y_i(\mathbf{w}^\top \mathbf{x}_i + b) \geq 1 - \xi_i,\quad \xi_i \geq 0
$$

Where:
- $\xi_i$: slack variables that allow violations
- $C$: penalty parameter that controls trade-off between margin size and misclassification

## Algorithm Steps

For **multiclass classification** using structured/vectorized SVM:

1. **Initialize Weight Matrix:** $\mathbf{W} \in \mathbb{R}^{K \times d}$ where each row corresponds to a class

2. **Compute Class Scores:** For sample $\mathbf{x}_i$, calculate $s = \mathbf{W} \mathbf{x}_i \in \mathbb{R}^K$

3. **Apply Multiclass Hinge Loss:**
   $$
   L_i = \sum_{j \neq y_i} \max(0, s_j - s_{y_i} + 1)
   $$

4. **Total Loss with Regularization:**
   $$
   \mathcal{L} = \frac{1}{N} \sum_{i=1}^N L_i + \frac{\lambda}{2} ||\mathbf{W}||^2
   $$

5. **Update Weights:** Use gradient descent on the hinge loss

## Advanced Topics

### Multiclass Classification Strategies

#### a) One-vs-All (OvA):
Train $K$ binary classifiers (for $K$ classes). For class $k$:
$$
y_i^{(k)} = \begin{cases}
+1 & \text{if } y_i = k \\
-1 & \text{otherwise}
\end{cases}
$$

The class with the highest score $\mathbf{w}_k^\top \mathbf{x} + b_k$ is predicted.

#### b) One-vs-One (OvO):
Train $\frac{K(K-1)}{2}$ classifiers for all class pairs. Prediction is made by majority vote among classifiers.

#### c) Structured Multiclass SVM:
Directly define a joint optimization problem using the multiclass hinge loss shown above. This approach is commonly used in practical implementations.

## Key Characteristics

### Advantages
- Effective in high-dimensional spaces
- Works well when there's a clear margin of separation  
- Memory efficient (only support vectors matter)
- Can model non-linear decision boundaries with kernels

### Limitations
- Doesn't scale well with large datasets (training time complexity is high)
- Choosing the right parameters can be tricky
- Not easily interpretable compared to decision trees or logistic regression

In [1]:
import numpy as np
from tqdm import tqdm
from cifar10.cifar10_utils import (
    get_all_data,
    get_test_data,
    extract_images_pca,
    normalize_data,
)

import sys
import os

project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(project_root)
from images.image_preprocessing import extract_raw_pixels, extract_color_histogram, extract_hog, extract_lbp

In [2]:
class LinearSVM:
    def __init__(self, input_dim, num_classes, lr=1e-3, reg=1e-4):
        """
        input_dim: data dimensionality (without bias)
        num_classes: number of classes
        lr: learning rate
        reg: L2 regularization strength
        """
        self.input_dim = input_dim + 1  # +1 for bias
        self.W = 0.001 * np.random.randn(
            num_classes, self.input_dim
        )  # weight initialization
        self.lr = lr
        self.reg = reg

    def compute_loss_and_gradients(self, X, y):
        """
        X: (batch_size, D) with bias column appended
        y: (batch_size,) labels
        Returns loss and gradient dW of shape (num_classes, D)
        """
        num_train = X.shape[0]
        scores = self.W.dot(X.T)  # class scores

        # correct class scores for each sample
        correct_scores = scores[y, np.arange(num_train)]

        # compute hinge loss margins
        margins = np.maximum(0, scores - correct_scores + 1.0)
        margins[y, np.arange(num_train)] = 0  # ignore correct class

        # compute total loss (data + regularization)
        data_loss = np.sum(margins) / num_train
        reg_loss = 0.5 * self.reg * np.sum(self.W * self.W)
        loss = data_loss + reg_loss

        # compute gradient
        binary = (margins > 0).astype(float)
        row_sum = np.sum(binary, axis=0)
        binary[y, np.arange(num_train)] = -row_sum  # contribution from correct class
        dW = binary.dot(X) / num_train  # average over batch
        dW += self.reg * self.W  # add regularization gradient

        return loss, dW

    def train(self, X, y, epochs=15, batch_size=256, verbose=True):
        """
        X: (N, original_input_dim) WITHOUT bias
        y: (N,) labels
        """
        X_bias = np.hstack([X, np.ones((X.shape[0], 1))])  # append bias
        N = X_bias.shape[0]

        for epoch in range(epochs):
            # shuffle data
            perm = np.random.permutation(N)
            X_shuffled = X_bias[perm]
            y_shuffled = y[perm]

            running_loss = 0.0
            num_batches = 0

            # mini-batch SGD
            for i in range(0, N, batch_size):
                X_batch = X_shuffled[i : i + batch_size]
                y_batch = y_shuffled[i : i + batch_size]

                loss, grad = self.compute_loss_and_gradients(X_batch, y_batch)
                self.W -= self.lr * grad  # update weights

                running_loss += loss
                num_batches += 1

            avg_loss = running_loss / num_batches
            if verbose:
                print(f"Epoch {epoch + 1}/{epochs}, avg_loss: {avg_loss:.4f}")

    def predict(self, X):
        """
        X: (num_samples, original_input_dim) WITHOUT bias
        Returns predicted labels array of shape (num_samples,)
        """
        X_bias = np.hstack([X, np.ones((X.shape[0], 1))])  # append bias
        scores = self.W.dot(X_bias.T)  # compute scores
        y_pred = np.argmax(scores, axis=0)  # choose class with max score
        return y_pred

In [ ]:
# Load data
x_train, y_train = get_all_data()
x_test, y_test = get_test_data()

# Preprocess: extract features and normalize
x_train = np.hstack(
    (
        extract_raw_pixels(x_train),
        extract_hog(x_train),
        extract_lbp(x_train),
        extract_color_histogram(x_train),
    )
)
x_train_norm, mean, std = normalize_data(x_train)
x_test = np.hstack(
    (
        extract_raw_pixels(x_test),
        extract_hog(x_test),
        extract_lbp(x_test),
        extract_color_histogram(x_test),
    )
)
x_test_norm = normalize_data(x_test, mean, std)


# Train algorithm
svm = LinearSVM(input_dim=x_train_norm.shape[1], num_classes=10, lr=1e-3, reg=1e-4)
svm.train(x_train_norm, y_train, epochs=10, batch_size=200)

# Evaluate
from sklearn.metrics import accuracy_score

y_pred = svm.predict(x_test_norm)
acc = accuracy_score(y_test, y_pred)
print(f"Test accuracy: {acc:.4f}")

Epoch 1/10, avg_loss: 3.1269
Epoch 2/10, avg_loss: 2.2137
Epoch 3/10, avg_loss: 2.0046
Epoch 4/10, avg_loss: 1.8907
Epoch 5/10, avg_loss: 1.8131
Epoch 6/10, avg_loss: 1.7506
Epoch 7/10, avg_loss: 1.7014
Epoch 8/10, avg_loss: 1.6609
Epoch 9/10, avg_loss: 1.6283
Epoch 10/10, avg_loss: 1.5981
Test accuracy: 0.6063
